<h1 style="text-align: center;"> Project name </h1>

<div style="display: flex; justify-content: space-around;">

<div style="width: 30%; text-align: center;">
<strong>Jie Zhao</strong>  
<br>
jiz273@g.harvard.edu
</div>

<div style="text-align: center; width: 80%; margin: 0 auto;">
    <strong>Abstract</strong><br>
    FER-2013 is a dataset of facial expressions that contains 35,887 grayscale images of faces with seven different emotions: anger, disgust, fear, happiness, sadness, surprise, and neutral. In this project, we explore various different approaches to classifying emotions, custom convolutional neural networks (CNNs), transfer learning and vision transformers. Prior work on this dataset without an auxiliary dataset ranges from 70-75% accuracy, where human classification accuracy is benchmarked between 60 and 70%. Most relied heavily on CNNs but in this project we also tested vision transformers. We use an autoencoder to filter out outlying images based on reconstruction data. Using this filtered dataset, our custom U-Net model with squeeze-excite blocks achieves a test accuracy of 64%, matching human benchmarks. Our most performant model is transfer learning with a ResNet50 model, which achieves a test accuracy of 68.5%, comparable to other papers. Other papers have found higher accuracies in the range of 73-78% but these were done with an auxiliary dataset or with a multi-label setup. Although novel, the vision transformer did not perform well, only marginally better than random. This is most likely due to the small sample size, since models with weaker inductive biases need more data to learn.
</div>

## Table of Contents

**1. [Introduction](#introduction)**  
 1.1 [Problem Statement](#problem-statement)  

**2. [Installation, Configuration and Set UP](#introduction)**  

**3. [Dataset and Data Preparation](#comprehensive-eda-review)**  
 2.1 [Data Description](#data-description)  
 2.2 [Explortaty Analysis]
 2.3 [Data Cleaning]
 2.4 [Data Analysis](#understand)  

**4. [Step-by-Step Project Development](#research-question)**  

**5. [Results and Demonstration](#baseline-model)**  

**6. [Uses and Benefits](#final-model)**  

**7. [Challenges / Lesson Learnt]

**8. [Youtube Video Links] 

**9. [References] 

**10. [Appendix] 



 ## 1. Introduction

### 1.1 Business Context

Food establishments in Boston are inspected to protect public health and ensure compliance with food safety regulations. These inspections generate large amounts of public data, including violation descriptions, inspection results, dates, locations, and business information.

However, this data is difficult for non-technical users to explore. Restaurant owners, city analysts, public health teams and general publics may not easily know:

which violations are increasing
which neighborhoods show higher inspection risk
which establishments have repeated issues
which violation types are most common
which past inspections are similar to a current problem

### 1.2 Problem Statement

## 2. Installation, Configuration and Set UP

2.1 Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pyarrow as pa




## 3. Dataset and Data Preparation

The Bostn Health Division of the Department of Inspectional Services ensures that all food establishments in the City of Boston meet relevant sanitary codes and standards. Businesses that serve food are inspected at least once a year, and follow-up inspections are performed on high risk establishments. Health inspections are also conducted in response to complaints of unsanitary conditions or illness. 

The Boston Food Establishment Inspections dataset contains the outcomes of food establishment inspections conducted in the city’s greater area since 2006. Updated daily, this dataset provides details about individual inspections and results of businesses serving food. For this analysis, we are working with a static version of the dataset, comprising 27 columns, and 884608 individual records.

The dataset includes information such as the time and location of each inspection, the business entity responsible, and the licensing details. It also records inspection outcomes, violations noted, and any follow-up actions or comments, offering a comprehensive view of Boston’s food safety practices.

3.1 Load data and inspect

In [2]:
#Load and inspect data
df = pd.read_csv('data/food_inspection_report_raw.csv')
df.info()

/var/folders/0x/_5wx0swx0vxgsyrkv847t75w0000gn/T/ipykernel_22344/3106629136.py:2: DtypeWarning: Columns (0: zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/food_inspection_report_raw.csv')


<class 'pandas.DataFrame'>
RangeIndex: 884608 entries, 0 to 884607
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   businessname  884608 non-null  str    
 1   dbaname       8431 non-null    str    
 2   legalowner    575201 non-null  str    
 3   namelast      884608 non-null  str    
 4   namefirst     508539 non-null  str    
 5   licenseno     884608 non-null  int64  
 6   issdttm       883712 non-null  str    
 7   expdttm       883928 non-null  str    
 8   licstatus     884608 non-null  str    
 9   licensecat    884608 non-null  str    
 10  descript      884608 non-null  str    
 11  result        884608 non-null  str    
 12  resultdttm    878210 non-null  str    
 13  violation     823807 non-null  str    
 14  viol_level    823807 non-null  str    
 15  violdesc      816937 non-null  str    
 16  violdttm      823804 non-null  str    
 17  viol_status   823807 non-null  str    
 18  status_date   3

In [3]:
#understand the data structure
print('size of data', df.shape)
print('columns', df.columns)
print('describe of data', df.describe())


size of data (884608, 26)
columns Index(['businessname', 'dbaname', 'legalowner', 'namelast', 'namefirst',
       'licenseno', 'issdttm', 'expdttm', 'licstatus', 'licensecat',
       'descript', 'result', 'resultdttm', 'violation', 'viol_level',
       'violdesc', 'violdttm', 'viol_status', 'status_date', 'comments',
       'address', 'city', 'state', 'zip', 'property_id', 'location'],
      dtype='str')
describe of data            licenseno    property_id
count  884608.000000  727234.000000
mean   107065.636450  150178.259790
std    144712.285047  103322.491624
min        54.000000       0.000000
25%     22153.000000   77703.000000
50%     28531.000000  155991.000000
75%    125523.000000  157956.000000
max    624593.000000  460598.000000


Explore potential meaningful columns for our analysis deeper to understand the type and cardinality

In [4]:
# violation and description columns
print('violcation code carinality\n\n', df['violation'].value_counts())
print('='*100)
print('violcation description carinality\n\n',df['violdesc'].value_counts())

violcation code carinality

 violation
23-4-602.13            43973
37-6-501.11-.12        39951
15-4-202.16            35183
36-6-501.11-.12        33806
08-3-305-307.11        30211
                       ...  
590.004/4-204.123-C        1
02-3-305.11(2)             1
590.003/3-801.11-C         1
                           1
590.005/5-402.14-PF        1
Name: count, Length: 462, dtype: int64
violcation description carinality

 violdesc
Non-Food Contact Surfaces Clean                                                                   43973
Improper Maintenance of Walls/Ceilings                                                            39951
Non-Food Contact Surfaces                                                                         35183
Improper Maintenance of Floors                                                                    33806
Food Protection                                                                                   30211
                                      

In [5]:
# more star means more severe
df['viol_level'].value_counts()

viol_level
*       579434
***     126544
**      110958
-         6869
1919         1
             1
Name: count, dtype: int64

| Raw value | Interpreted meaning   | Action                    |
| --------- | --------------------- | ------------------------- |
| `*`       | Low severity          | map to severity_score = 1 |
| `**`      | Medium severity       | map to severity_score = 2 |
| `***`     | High severity         | map to severity_score = 3 |
| `-`       | Missing / unspecified | null                      |
| `1919`    | malformed data        | remove/null               |


In [6]:
# require mapping violation level from stars to understanable severity, and assign numerical scores to severity
severity_map = {
    "*": "low",
    "**": "medium",
    "***": "high",
    "-": None,
    "1919": None
}
severity_score_map= {
    "*": 1,
    "**": 2,
    "***": 3
}


In [7]:
df['viol_status'].value_counts()

viol_status
Fail    452084
Pass    365983
          5740
Name: count, dtype: int64

In [8]:
df['result'].value_counts()

result
HE_Fail       369707
HE_Pass       282095
HE_Filed       92851
HE_FailExt     73291
HE_Hearing     27393
HE_NotReq      23985
HE_TSOP         7669
HE_VolClos      2839
HE_OutBus       2521
Pass             964
HE_Closure       711
Fail             238
HE_FAILNOR       146
HE_Misc          128
DATAERR           42
HE_Hold           17
Failed             6
Closed             2
PassViol           2
NoViol             1
Name: count, dtype: int64

In [9]:
df[df['result']=='PassViol']

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,...,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
492580,MELO'S GROCERY,NaN,ESPAILLAT MANUEL A,GUZMAN,ORQUIDIA,24704,2009-04-15 18:29:55+00,2008-01-01 04:59:00+00,Inactive,RF,...,NaN,NaN,NaN,NaN,331 CENTRE ST,JAMAICA PLAIN,MA,2130,27981.0,"(42.32287000024958, -71.10500000140604)"
753673,Taj Boston,NaN,MPE HOTEL I LLC,ALBRIGHT,MAUREEN,24892,2012-01-11 13:47:28+00,2017-01-01 04:59:00+00,Inactive,RF,...,NaN,NaN,NaN,NaN,15 ARLINGTON ST,BOSTON,MA,2116.0,4827.0,"(42.35282999954591, -71.07160000160395)"


Each record corresponds to a single violation, meaning that multiple records can exist for the same inspection if several violations are found during the procedure. Each violation is documented with its own description and severity classification, but records from the same inspection share the inspection result and date-time fields.

In [10]:
pd.set_option('display.max_columns', None) 

In [11]:
df.head(15)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,descript,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clea...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,NaN,Several dented cans found on storage shelves. ...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,NaN,Wet wiping cloths found on counter tops . Remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
5,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2019-02-04 18:06:00+00,02-3-602.11-.12/3-302.12,*,Food Container Labels,2019-02-04 18:06:00+00,Fail,NaN,No labels on bulk containers . Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
6,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2019-02-04 18:06:00+00,10-3-304.12,*,Food Utensil Storage,2019-02-04 18:06:00+00,Fail,NaN,Scoops found submerges in flour and other bulk...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
7,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_NotReq,2022-03-15 19:22:42.093+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
8,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Pass,2017-08-11 14:10:25+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
9,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Pass,2017-12-15 18:58:58+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


Explore comments column, we can see the comments usually include the observation of violation, and improvement suggestion.

In [12]:
pd.set_option('display.max_colwidth', None)
df['comments'][:8]

0                                                                               One staff person without hair restraint. Provide
1                                                                     Caked on food debris on can opener blade. Clean to remove.
2                      Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.
3    Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.
4                                      Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.
5                                                                                         No labels on bulk containers . Provide
6                                      Scoops found submerges in flour and other bulk container. Store properly with handles up.
7                                                                                                

In [13]:
#description of inspection result
ref = pd.read_csv('data/InspectionResult_description.csv')
# Standardize column names
ref.columns = ref.columns.str.lower().str.strip()
ref

,inspectionresult,inferreddescription
0,HE_Fail,Inspection failed; violations found
1,HE_Pass,Inspection passed; no issues found
2,HE_Filed,Minor violations found; no urgent follow-up required
3,HE_FailExt,Extended failure from prior inspection
4,HE_Hearing,Violations being addressed; follow-up required
5,HE_NotReq,Inspection not required; no immediate need
6,HE_TSOP,Temporary suspension of permit issued
7,HE_OutBus,Business closed; out of operation
8,HE_VolClos,Voluntary closure by business to avoid penalties
9,HE_Closure,Forced closure due to critical violations


We aggregate inspection results into category and assign risk scores

In [14]:
result_group_map = {
    "HE_Pass": "pass",
    "Pass": "pass",
    
    "HE_Filed": "minor_violation",

    "HE_Fail": "fail",
    "Fail": "fail",
    "Failed": "fail",
    "HE_FAILNOR": "fail",

    "HE_FailExt": "extended_fail",
    "HE_Hearing": "hearing",

    "HE_TSOP": "temporary_suspension",
    "HE_VolClos": "voluntary_closure_avoid",
    "HE_Closure": "forced_closure",

    "HE_OutBus": "out_of_business",
    "Closed": "closed",

    "HE_NotReq": "not_required",
    "HE_Misc": "misc",
    "DATAERR": "data_error",
    "HE_Hold": "hold"
}

risk_score_map = {
    "pass": 1,
    "minor_violation": 2,
    "hold": 2,

    "fail": 3,
    "extended_fail": 3,
    "hearing": 3,

    "temporary_suspension": 4,
    "voluntary_closure_avoid": 4,
    "forced_closure": 4,

    "out_of_business": None,
    "closed": None,
    "not_required": None,
    "misc": None,
    "data_error": None,
}



#upon initial inspection of the data, we decided the columns below are relative and meaningful for our anlaysis.

- businessname: business name
- licenseno: Key identifier for the business/ restaurant
-result: result of  inspection
-resultdttm: date on which the results were generated
-violation: coding of law regulation related to violations
- viol_level: level of violation
-violdesc: reason of violation
- violdttm: date on which violation status was generated
- viol_status: status for violation: Fail or Pass
-status_date: date on which violation status was set to pass
- comments: comments given to the food establishment for improvement
- address: address of the business
- zipcode: zipcode of the business
- location: latitude and longitude of the business location

In [15]:
cols = [
    "businessname",
    "licenseno",
    "result",
    "resultdttm",
    "violation",
    "viol_level",
    "violdesc",
    "violdttm",
    "viol_status",
    "comments",
    "address",
    "zip",
    "location",
]

data = df[cols].copy()
data.head()

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"


In [16]:
data["severity_level"] = data["viol_level"].map(severity_map)
data["severity_score"] = data["viol_level"].map(severity_score_map)

data["result_group"] = data["result"].map(result_group_map)
data["risk_score"] = data["result_group"].map(risk_score_map)

In [17]:
# merge the description of inspection result to the main dataframe
data = data.merge(
    ref[["inspectionresult", "inferreddescription"]],
    left_on="result",
    right_on="inspectionresult",
    how="left"
)
data = data.drop(columns=["inspectionresult"])

In [18]:
# the analysis focuses on violation related content, we drop the rows with missing violation, result, viol_level, viol_status
data.dropna(subset=['violation','result','viol_level','viol_status'], inplace=True)


823,807 usable rows,1-2% percent of the data is missing.

In [19]:
data.info()

<class 'pandas.DataFrame'>
Index: 823807 entries, 0 to 884607
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   businessname         823807 non-null  str    
 1   licenseno            823807 non-null  int64  
 2   result               823807 non-null  str    
 3   resultdttm           820159 non-null  str    
 4   violation            823807 non-null  str    
 5   viol_level           823807 non-null  str    
 6   violdesc             816937 non-null  str    
 7   violdttm             823804 non-null  str    
 8   viol_status          823807 non-null  str    
 9   comments             797312 non-null  str    
 10  address              823678 non-null  str    
 11  zip                  823322 non-null  object 
 12  location             753462 non-null  str    
 13  severity_level       816936 non-null  str    
 14  severity_score       816936 non-null  float64
 15  result_group         823807 non-n

In [20]:
# handling na
# need date time data for analysis 
data["resultdttm"] = pd.to_datetime(df["resultdttm"], errors="coerce")
data["resultdttm"] = pd.to_datetime(data["resultdttm"], errors="coerce")
data["year"] = data["resultdttm"].dt.year
data["month"] = data["resultdttm"].dt.strftime("%Y-%m")
data = data.dropna(subset=["resultdttm"])

# fill na with unknown
data["violdesc"] = data["violdesc"].fillna("Unknown violation")
data["severity_level"] = data["severity_level"].fillna("Unknown severity_level")
data["comments"] = data["comments"].fillna("No comment")
data["address"] = data["address"].fillna("Unknown")
data["inferreddescription"] = data["inferreddescription"].fillna("Unknown inspection outcome")
data["address"] = data["address"].fillna("Unknown")
data["zip"] = data["zip"].astype("string")


In [21]:
data.info()

<class 'pandas.DataFrame'>
Index: 820092 entries, 0 to 884607
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   businessname         820092 non-null  str                
 1   licenseno            820092 non-null  int64              
 2   result               820092 non-null  str                
 3   resultdttm           820092 non-null  datetime64[us, UTC]
 4   violation            820092 non-null  str                
 5   viol_level           820092 non-null  str                
 6   violdesc             820092 non-null  str                
 7   violdttm             820090 non-null  str                
 8   viol_status          820092 non-null  str                
 9   comments             820092 non-null  str                
 10  address              820092 non-null  str                
 11  zip                  819618 non-null  string             
 12  location          

In [22]:
data.head(2)

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription,year,month
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",medium,2.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03


In [ ]:
#write it to parquet file, more efficient for database use

data.to_parquet("data/food_inspections_clean.parquet", engine="pyarrow",index=False)

3. DuckDB setup

Load cleaned data into DuckDB

In [34]:
import duckdb

con = duckdb.connect("data/food_inspections_clean.duckdb")

con.execute("""
CREATE OR REPLACE TABLE inspections AS
SELECT *
FROM read_parquet('data/food_inspections_clean.parquet')
""")

con.execute("SELECT COUNT(*) FROM inspections").fetchall()

[(820092,)]

4. SQL functions 

In [35]:
def top_violation_types(limit=15):
    query = """
    SELECT 
        violdesc,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE violdesc IS NOT NULL
    GROUP BY violdesc
    ORDER BY violation_count DESC
    LIMIT ?
    """
    return con.execute(query, [limit]).df()

Test SQL results

In [36]:
top_violation_types()

,violdesc,violation_count
0,Non-Food Contact Surfaces Clean,43636
1,Improper Maintenance of Walls/Ceilings,39837
2,Non-Food Contact Surfaces,34952
3,Improper Maintenance of Floors,33684
4,Food Protection,30033
5,Food Contact Surfaces Clean,26179
6,Hand Cleaner Drying Tissue Signage,23162
7,Premises Maintained,22067
8,Wiping Cloths Clean Sanitize,18159
9,Installed and Maintained,17575


Aggregated data would usually surface interesting insights, we would like have understanding high level around questions below. 

•	Which neighborhoods have highest violation rates? 
•	Which cuisines fail most often? 
•	Which violations are increasing? 
•	Which restaurants repeatedly fail? 
•	Which months show spikes? 


5. Chart functions 

6. Deep learning / embeddings

7. RAG retrieval

8. Router

9. LLM summary

10. Final demo function

In [ ]:
def ask(question):
    route = route_question(question)

    if route == "top":
        df = top_violation_types()
        plot_bar(...)
        return df

    elif route == "trend":
        df = violation_trend()
        plot_line(...)
        return df

    elif route == "semantic":
        return retrieve_similar_violations(question)

Demo:

ask("What are the most common violations?")
ask("Show violation trend over time")
ask("Find violations similar to refrigeration problems")

Reference

https://data.boston.gov/dataset/food-establishment-inspections